## Load the required packages

In [1]:
import anndata
import pandas as pd
from scipy.io import mmwrite
import os
import scanpy as sc
from scipy.sparse import coo_matrix, csr_matrix
from pathlib import Path
import numpy as np

## Set path

In [2]:
working_dir = Path(os.getcwd())
dataset_dir = f"{working_dir.parents[1]}/datasets/CITE-SLN-Gayoso"
base_dir = f"{working_dir.parents[1]}/outputs/batch variation/208_M2-208_M1/scMM"
os.makedirs(base_dir, exist_ok=True)

## Load data

In [3]:
train_data = anndata.read(f"{dataset_dir}/SLN208_M2.h5ad")
test_data = anndata.read(f"{dataset_dir}/SLN208_M1.h5ad")
train_data, test_data

(AnnData object with n_obs × n_vars = 7105 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 AnnData object with n_obs × n_vars = 8715 × 13553
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression')

## Select common genes

In [4]:
common_genes = np.intersect1d(train_data.var.index, test_data.var.index)
train_data = train_data[:, common_genes]
test_data = test_data[:, common_genes]

## Select common proteins

In [5]:
common_proteins = np.intersect1d(np.array(train_data.uns["protein_names"]), np.array(test_data.uns["protein_names"]))
if isinstance(train_data.obsm["protein_expression"], pd.DataFrame):
    train_protein_expression = train_data.obsm["protein_expression"]
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = train_protein_expression
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)
else:
    train_protein_expression = pd.DataFrame(train_data.obsm["protein_expression"].todense(), index=train_data.obs.index, columns=train_data.uns["protein_names"])
    train_protein_expression = train_protein_expression.loc[:, common_proteins]
    train_data.obsm["protein_expression"] = csr_matrix(train_protein_expression.values)
    train_data.uns["protein_names"] = np.array(train_protein_expression.columns)

if isinstance(test_data.obsm["protein_expression"], pd.DataFrame):
    test_protein_expression = test_data.obsm["protein_expression"]
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = test_protein_expression
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)
else:
    test_protein_expression = pd.DataFrame(test_data.obsm["protein_expression"].todense(), index=test_data.obs.index, columns=test_data.uns["protein_names"])
    test_protein_expression = test_protein_expression.loc[:, common_proteins]
    test_data.obsm["protein_expression"] = csr_matrix(test_protein_expression.values)
    test_data.uns["protein_names"] = np.array(test_protein_expression.columns)

## Select highly variable genes

In [6]:
data = anndata.concat([train_data, test_data], axis=0)
sc.pp.normalize_total(data, target_sum=1e4)
sc.pp.log1p(data)
sc.pp.highly_variable_genes(data, n_top_genes=5000)
index = data.var['highly_variable'].values

train_data = train_data[:, index]
test_data = test_data[:, index]
train_data, test_data, train_data.shape, test_data.shape

(View of AnnData object with n_obs × n_vars = 7105 × 5000
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 View of AnnData object with n_obs × n_vars = 8715 × 5000
     obs: 'n_protein_counts', 'n_proteins', 'seurat_hash_id', 'batch_indices', 'hash_id', 'n_genes', 'percent_mito', 'leiden_subclusters', 'cell_types'
     var: 'gene_ids', 'feature_types', 'highly_variable', 'highly_variable_mean_variance', 'encode', 'hvg_encode'
     uns: 'protein_names', 'version'
     obsm: 'protein_expression',
 (7105, 5000),
 (8715, 5000))

In [7]:
train_gene_expression = coo_matrix(train_data.X).T
train_protein_expression = coo_matrix(train_protein_expression.values).T
train_cells = pd.DataFrame(train_data.obs.index)
train_genes = pd.DataFrame(train_data.var.index)
train_proteins = pd.DataFrame(train_data.uns["protein_names"])

In [8]:
test_gene_expression = coo_matrix(test_data.X).T
test_protein_expression = coo_matrix(test_protein_expression.values).T
test_cells = pd.DataFrame(test_data.obs.index)
test_genes = pd.DataFrame(test_data.var.index)
test_proteins = pd.DataFrame(test_data.uns["protein_names"])

## Create the saved format required to run scMM

In [9]:
train_RNA_dir = f"{base_dir}/train_RNA-seq"
train_protein_dir = f"{base_dir}/train_CITE-seq"
os.makedirs(train_RNA_dir, exist_ok=True)
os.makedirs(train_protein_dir, exist_ok=True)

test_RNA_dir = f"{base_dir}/test_RNA-seq"
test_protein_dir = f"{base_dir}/test_CITE-seq"
os.makedirs(test_RNA_dir, exist_ok=True)
os.makedirs(test_protein_dir, exist_ok=True)

In [10]:
mmwrite(f"{train_RNA_dir}/RNA_count.mtx", train_gene_expression)
mmwrite(f"{train_protein_dir}/Protein_count.mtx", train_protein_expression)
train_cells.to_csv(f"{train_RNA_dir}/barcode.tsv", sep="\t", index=False, header=False)
train_cells.to_csv(f"{train_protein_dir}/barcode.tsv", sep="\t", index=False, header=False)
train_genes.to_csv(f"{train_RNA_dir}/gene.tsv", sep="\t", index=False, header=False)
train_proteins.to_csv(f"{train_protein_dir}/protein.tsv", sep="\t", index=False, header=False)

In [11]:
mmwrite(f"{test_RNA_dir}/RNA_count.mtx", test_gene_expression)
mmwrite(f"{test_protein_dir}/Protein_count.mtx", test_protein_expression)
test_cells.to_csv(f"{test_RNA_dir}/barcode.tsv", sep="\t", index=False, header=False)
test_cells.to_csv(f"{test_protein_dir}/barcode.tsv", sep="\t", index=False, header=False)
test_genes.to_csv(f"{test_RNA_dir}/gene.tsv", sep="\t", index=False, header=False)
test_proteins.to_csv(f"{test_protein_dir}/protein.tsv", sep="\t", index=False, header=False)